In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_38_try_1.pickle

In [ ]:
%%PandasProfile
### cell 39 ###

def grab_subset_of_data_51(df, question_of_interest):
    # Select only the columns containing the question text
    sub = df.filter(like=question_of_interest, axis=1)
    # Rename by stripping off the prefix and leading whitespace in one vectorized step
    new_cols = sub.columns.str.split('-', n=1).str[-1].str.lstrip()
    sub.columns = new_cols
    # Drop rows where all of these columns are NaN
    return sub.dropna(how='all')

question_of_interest_cell51 = \
    'Which of the following natural language processing (NLP) methods do you use on a regular basis?'

# Helper to compute counts and percentages for a given year
def _compute_yearly(df, year_tag, df_var):
    nlp = grab_subset_of_data_51(df, question_of_interest_cell51)
    counts = nlp.count().sort_values()
    percentages = counts.mul(100).div(len(nlp))
    # Identify the "Transformer" entry by matching 'BERT' and rename it
    transformer_label = percentages.index[percentages.index.str.contains('BERT', na=False)][0]
    pct_cell51 = percentages.rename({transformer_label: 'Transformers'})[['Transformers']]
    # Assign into the same variable names as before
    globals()[f'nlp_df_{year_tag}_cell51'] = nlp
    globals()[f'counts_{year_tag}'] = counts
    globals()[f'percentages_{year_tag}'] = percentages
    globals()[f'percentages_{year_tag}_cell51'] = pct_cell51

# Compute for each year
_compute_yearly(responses_df_2022_cell10, '2022', 'nlp_df_2022_cell51')
_compute_yearly(responses_df_2021,         '2021', 'nlp_df_2021')
_compute_yearly(responses_df_2020,         '2020', 'nlp_df_2020')
_compute_yearly(responses_df_2019_cell10,  '2019', 'nlp_df_2019_cell10')

# Display info
(percentages_2019_cell51.info(), 
 percentages_2020_cell51.info(), 
 percentages_2021_cell51.info(), 
 percentages_2022_cell51.info())